#  WESAD Stress Detection — CNN-LSTM · LOSO
**Binary classification:** Baseline (0) vs Stress (1)  
**Evaluation:** Leave-One-Subject-Out cross-validation  
**Architecture:** 2× Conv1D (spatial) → LSTM (temporal) → Dense  
**Bonus:** Optional Sensor Self-Attention variant included  

---
### Before you start
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Upload `WDM_dataset/Features/` to your Google Drive
3. Run cells **in order**
---

In [ ]:
# ── Cell 1 · Environment Setup ─────────────────────────────────────────────
# Runtime → Change runtime type → T4 GPU → Save   (before running anything)

from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
# ── Cell 2 · Paths & Directories ───────────────────────────────────────────
#  Only edit BASE_PATH if your Drive folder has a different name/location.
# Expected layout:
#   MyDrive/WDM_dataset/Features/chest/all_chest_*.parquet
#   MyDrive/WDM_dataset/Features/wrist/all_wrist_*.parquet

from pathlib import Path

BASE_PATH  = '/content/drive/MyDrive/Features'
OUTPUT_DIR = Path('/content/drive/MyDrive/dl_results')
MODEL_DIR  = Path('/content/drive/MyDrive/dl_models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Quick sanity check
try:
    found = [f.name for f in Path(BASE_PATH).iterdir()]
    print(" Dataset found:", found)
except FileNotFoundError:
    print(" Path not found — update BASE_PATH above.")


 Dataset found: ['S3', 'S4', 'S5', 'S15', 'S6', 'chest', 'S7', 'S10', 'S2', 'S14', 'S17', 'S16', 'S9', 'S11', 'wrist', 'S8', 'S13']


In [ ]:
# ── Cell 3 · Imports ────────────────────────────────────────────────────────
import gc, json, traceback, warnings
import numpy  as np
import pandas as pd
from   pathlib import Path

import tensorflow as tf
from   tensorflow import keras
from   tensorflow.keras import layers

from sklearn.preprocessing    import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

warnings.filterwarnings('ignore')
print(" All imports OK")


 All imports OK


In [ ]:
# ── Cell 4 · Data Loader ────────────────────────────────────────────────────
# Reads the all_<modality>_<type>.parquet files and merges them.
# Label mapping: 1 (Baseline) → 0,  2 (Stress) → 1
# Meditation (4) and Amusement (3) are dropped so this is binary classification.

MERGE_KEYS       = ['subject', 'window_idx', 'start_time', 'end_time', 'label']
NON_FEATURE_COLS = set(MERGE_KEYS)


def load_dataset(config: dict) -> pd.DataFrame:
    """
    config = {'chest': ['acc', 'eda'], 'wrist': ['acc']}
    Each key is a modality folder; values are feature-bundle names.
    Returns merged DataFrame with label 0=baseline, 1=stress.
    """
    all_dfs = []
    for modality, bundles in config.items():
        dfs = []
        for bundle in bundles:
            path = Path(BASE_PATH) / modality / f'all_{modality}_{bundle}.parquet'
            if not path.exists():
                raise FileNotFoundError(f"Missing: {path}")
            dfs.append(pd.read_parquet(path))

        merged = dfs[0]
        for df in dfs[1:]:
            merged = merged.merge(df, on=MERGE_KEYS)
        all_dfs.append(merged)

    final = all_dfs[0]
    for df in all_dfs[1:]:
        final = final.merge(df, on=MERGE_KEYS)

    # Binary: keep only Baseline(1) and Stress(2)
    final = final[final['label'].isin([1, 2])].copy()
    final['label'] = final['label'].map({1: 0, 2: 1})
    return final


print("Data loader ready")


Data loader ready


In [ ]:
# ── Cell 5 · Shared Preprocessing Utilities ─────────────────────────────────
# Used by every LOSO fold regardless of model type.
# Rules:
#   • NaN imputation  → median of TRAINING windows only (no test leakage)
#   • Scaling         → StandardScaler fit on training windows only
#   • NaN/Inf cleanup → replace residual bad values with 0

def get_feature_names(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c not in NON_FEATURE_COLS]


def impute_and_scale(X_train_raw, X_test_raw):
    """
    1. Median-impute NaNs (train medians applied to both splits)
    2. StandardScaler (fit on train, transform both)
    3. Replace any residual NaN/Inf with 0
    Returns: X_train_scaled, X_test_scaled, scaler, medians
    """
    medians = np.nanmedian(X_train_raw, axis=0)

    for arr in [X_train_raw, X_test_raw]:
        mask = np.isnan(arr)
        if mask.any():
            arr[mask] = np.take(medians, np.where(mask)[1])

    scaler      = StandardScaler()
    X_train_sc  = scaler.fit_transform(X_train_raw)
    X_test_sc   = scaler.transform(X_test_raw)

    X_train_sc  = np.nan_to_num(X_train_sc, nan=0.0, posinf=0.0, neginf=0.0)
    X_test_sc   = np.nan_to_num(X_test_sc,  nan=0.0, posinf=0.0, neginf=0.0)
    return X_train_sc, X_test_sc, scaler, medians


print(" Shared preprocessing ready")


 Shared preprocessing ready


In [ ]:
# ── Cell 6 · Metrics Utility ────────────────────────────────────────────────
def evaluate(y_true, probs, threshold=0.5):
    """Returns a dict of all key metrics for one LOSO fold."""
    preds = (probs >= threshold).astype(int)

    try:
        auc = roc_auc_score(y_true, probs)
    except ValueError:
        auc = float('nan')   # only one class in fold (rare)

    cm          = confusion_matrix(y_true, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return dict(
        accuracy    = accuracy_score(y_true, preds),
        precision   = precision_score(y_true, preds, average='binary', zero_division=0),
        recall      = recall_score(y_true, preds, average='binary', zero_division=0),
        f1          = f1_score(y_true, preds, average='binary', zero_division=0),
        specificity = specificity,
        roc_auc     = auc,
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
    )


def print_fold_summary(metrics_df):
    """Prints the global confusion matrix and mean metrics — same format as loso.py."""
    print("\n═══ GLOBAL CONFUSION MATRIX (sum over all folds) ═══")
    for k, label in [('tp','True Positives'), ('tn','True Negatives'),
                     ('fp','False Positives'), ('fn','False Negatives')]:
        print(f"  {label:<18}: {int(metrics_df[k].sum())}")
    print("\n═══ MEAN METRICS ACROSS SUBJECTS ═══")
    cols = ['accuracy','precision','recall','specificity','f1','roc_auc']
    print(metrics_df[cols].mean().round(4).to_string())


print(" Metrics utility ready")


 Metrics utility ready


In [ ]:
# ── Cell 7 · Input Reshaping for CNN-LSTM ───────────────────────────────────
# The hybrid model has two stages:
#   Stage 1 — CNN extracts spatial features from the sensor-group axis
#   Stage 2 — LSTM reads the CNN-extracted feature sequence temporally
#
# Input shape: (N, n_sensor_groups, max_features_per_group)
# This is identical to the pure CNN notebook — Conv1D runs across groups,
# then LSTM reads the resulting feature maps as a sequence of timesteps.

SENSOR_PREFIXES = ['acc', 'eda', 'emg', 'heart', 'temp', 'bvp', 'resp', 'scl', 'scr']


def group_by_sensor(feature_names: list) -> dict:
    groups = {}
    for col in feature_names:
        matched = next((p for p in SENSOR_PREFIXES if col.lower().startswith(p)), None)
        key = matched if matched else 'other'
        groups.setdefault(key, []).append(col)
    return groups


def to_3d(X_2d: np.ndarray, feature_names: list):
    groups  = group_by_sensor(feature_names)
    g_names = list(groups.keys())
    T       = max(len(v) for v in groups.values())
    c_idx   = {n: i for i, n in enumerate(feature_names)}
    X_3d    = np.zeros((X_2d.shape[0], len(g_names), T), dtype=np.float32)
    for gi, gn in enumerate(g_names):
        for fi, col in enumerate(groups[gn]):
            X_3d[:, gi, fi] = X_2d[:, c_idx[col]]
    return X_3d, groups


def prepare_input(X_scaled, feature_names):
    X_3d, _ = to_3d(X_scaled, feature_names)
    return X_3d


print(" CNN-LSTM reshaping ready")


 CNN-LSTM reshaping ready


In [ ]:
# ── Cell 8 · CNN-LSTM Architecture ──────────────────────────────────────────
# CNN-LSTM = CNN feature extractor + LSTM temporal reasoner
#
# HOW THE TWO STAGES CONNECT:
#   Conv1D processes the (n_groups, features) "image" → outputs (n_groups, 128)
#   This is a sequence of 128-dimensional vectors, one per sensor group.
#   LSTM then reads this sequence group-by-group, learning which GROUP ORDER
#   matters (e.g. "EDA rises → then HR rises → then EMG fires" = stress).
#
# Why this beats pure CNN or pure LSTM?
#   CNN learns intra-group AND inter-group local patterns well.
#   LSTM learns longer-range group-order dynamics.
#   Together they capture both spatial and sequential structure.

EPOCHS     = 100
BATCH_SIZE = 32
PATIENCE   = 12
MODEL_TAG  = 'cnn_lstm'


def build_model(input_shape: tuple) -> keras.Model:
    """input_shape = (n_groups, max_features_per_group)"""
    inp = keras.Input(shape=input_shape)

    # ── CNN block (spatial feature extraction) ────────────────────────────
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    # x shape here: (batch, n_groups, 128)

    # ── LSTM block (temporal reasoning over CNN feature maps) ─────────────
    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dropout(0.4)(x)

    # ── Classification head ───────────────────────────────────────────────
    x   = layers.Dense(64, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc'), 'accuracy'],
    )
    return model


_m = build_model((7, 30))
_m.summary()
del _m
print("\n CNN-LSTM builder ready")


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 7, 30)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 7, 64)          │         5,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 7, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 7, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 7, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 7, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 84,929 (331.75 KB)

 Trainable params: 84,545 (330.25 KB)

 Non-trainable params: 384 (1.50 KB)


 CNN-LSTM builder ready


In [ ]:
# ── Cell 9 · (Optional) Attention-Enhanced CNN-LSTM ─────────────────────────
# Drop-in replacement for build_model() above.
# Adds a self-attention layer between CNN and LSTM so the model can
# explicitly weight which sensor groups are most informative per sample.
#
# Use this if baseline CNN-LSTM results are underwhelming.
# It trains ~30% slower on Colab CPU but often gains 1-3% F1.


class SensorAttention(layers.Layer):
    """
    Dot-product self-attention over the sensor-group axis.
    Input:  (batch, n_groups, channels)
    Output: (batch, n_groups, channels)  — same shape, reweighted
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        dim = input_shape[-1]
        self.W_q = self.add_weight(shape=(dim, dim), initializer='glorot_uniform',
                                    trainable=True, name='Wq')
        self.W_k = self.add_weight(shape=(dim, dim), initializer='glorot_uniform',
                                    trainable=True, name='Wk')
        self.W_v = self.add_weight(shape=(dim, dim), initializer='glorot_uniform',
                                    trainable=True, name='Wv')
        super().build(input_shape)

    def call(self, x):
        Q = tf.matmul(x, self.W_q)   # (B, G, D)
        K = tf.matmul(x, self.W_k)
        V = tf.matmul(x, self.W_v)
        scale   = tf.math.sqrt(tf.cast(tf.shape(K)[-1], tf.float32))
        scores  = tf.matmul(Q, K, transpose_b=True) / scale   # (B, G, G)
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, V)   # (B, G, D)


def build_model_with_attention(input_shape: tuple) -> keras.Model:
    inp = keras.Input(shape=input_shape)

    x = layers.Conv1D(64,  kernel_size=3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)

    # Attention layer reweights sensor groups
    x = SensorAttention()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy',
                  metrics=[keras.metrics.AUC(name='auc'), 'accuracy'])
    return model


#  To use attention: replace build_model with build_model_with_attention
# in the LOSO loop below.  e.g.:  model = build_model_with_attention(...)
print(" Attention layer ready  (optional — swap in the LOSO loop)")


✅ Attention layer ready  (optional — swap in the LOSO loop)


In [ ]:
# ── Cell 10 · LOSO Training Loop ────────────────────────────────────────────


def run_loso(df: pd.DataFrame, exp_name: str,
             use_attention: bool = False,
             fold_save_dir: Path = None) -> tuple:
    subjects, metrics_rows, pred_rows = sorted(df['subject'].unique()), [], []
    _builder = build_model_with_attention if use_attention else build_model

    for fold_i, test_sub in enumerate(subjects):
        print(f"\nFold {fold_i+1}/{len(subjects)}  test={test_sub}", flush=True)

        train_df = df[df['subject'] != test_sub]
        test_df  = df[df['subject'] == test_sub]
        win_idx  = test_df['window_idx'].values
        feat     = get_feature_names(train_df)

        X_tr_raw = train_df[feat].values.astype(np.float32)
        X_te_raw = test_df[feat].values.astype(np.float32)
        y_tr     = train_df['label'].values.astype(np.int32)
        y_te     = test_df['label'].values.astype(np.int32)

        X_tr, X_te, _, _ = impute_and_scale(X_tr_raw, X_te_raw)

        X_tr_3d, _ = to_3d(X_tr, feat)
        X_te_3d, _ = to_3d(X_te, feat)
        print(f"  shape={X_tr_3d.shape[1:]}  train={len(y_tr)}  test={len(y_te)}")

        classes = np.unique(y_tr)
        cw      = dict(zip(classes.tolist(),
                           compute_class_weight('balanced', classes=classes, y=y_tr).tolist()))

        tf.keras.backend.clear_session()
        model = _builder(X_tr_3d.shape[1:])

        cbs = [
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                          restore_best_weights=True, verbose=0),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                              patience=6, min_lr=1e-6, verbose=0),
        ]
        if fold_save_dir:
            cbs.append(keras.callbacks.ModelCheckpoint(
                str(fold_save_dir / f'fold_{test_sub}.weights.h5'),
                monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=0))

        hist = model.fit(X_tr_3d, y_tr, epochs=EPOCHS, batch_size=BATCH_SIZE,
                         validation_split=0.1, class_weight=cw,
                         callbacks=cbs, verbose=0, shuffle=True)
        print(f"  stopped epoch {len(hist.history['loss'])}/{EPOCHS}")

        probs = model.predict(X_te_3d, verbose=0).ravel()
        m     = evaluate(y_te, probs)
        print(f"  Acc={m['accuracy']:.4f}  F1={m['f1']:.4f}  "
              f"AUC={m['roc_auc']:.4f}  TP={m['tp']}  FP={m['fp']}  "
              f"FN={m['fn']}  TN={m['tn']}")

        m.update(subject=test_sub)
        metrics_rows.append(m)
        for i, (yt, p) in enumerate(zip(y_te, probs)):
            pred_rows.append(dict(model=MODEL_TAG, experiment=exp_name,
                                  subject=test_sub, window_idx=win_idx[i],
                                  y_true=int(yt), y_prob=float(p)))
        del model; gc.collect()

    mdf = pd.DataFrame(metrics_rows)
    print_fold_summary(mdf)
    return mdf, pd.DataFrame(pred_rows)


print(" CNN-LSTM LOSO runner ready")


 CNN-LSTM LOSO runner ready


In [ ]:
# ── Cell 11 · Run Experiments ───────────────────────────────────────────────
#   USE_ATTENTION = True  → uses the Sensor-Attention CNN-LSTM variant
#     USE_ATTENTION = False → uses the standard CNN-LSTM

QUICK_TEST     = False    #
USE_ATTENTION  = False   #

EXPERIMENTS = {
    'chest_acc':         {'chest': ['acc']},
    'chest_eda':         {'chest': ['eda']},
    'chest_emg':         {'chest': ['emg']},
    'chest_heart':       {'chest': ['heart']},
    'chest_statistical': {'chest': ['statistical']},
    'wrist_acc':         {'wrist': ['acc']},
    'wrist_eda':         {'wrist': ['eda']},
    'wrist_heart':       {'wrist': ['heart']},
    'wrist_statistical': {'wrist': ['statistical']},
    'chest_all':  {'chest': ['acc', 'eda', 'emg', 'heart', 'statistical']},
    'wrist_all':  {'wrist': ['acc', 'eda', 'heart', 'statistical']},
    'multimodal': {'chest': ['acc', 'eda', 'emg', 'heart', 'statistical'],
                   'wrist': ['acc', 'eda', 'heart', 'statistical']},
}

if QUICK_TEST:
    EXPERIMENTS = {'chest_acc': {'chest': ['acc']}}
    print("⚡ Quick test mode")
else:
    attn_tag = "with attention" if USE_ATTENTION else "standard"
    print(f"Full grid ({attn_tag}): {len(EXPERIMENTS)} experiments")

all_metrics, all_predictions = [], []

for exp_name, config in EXPERIMENTS.items():
    print(f"\n{'='*55}\n  {MODEL_TAG.upper()} · {exp_name}\n{'='*55}")
    try:
        df = load_dataset(config)
        if QUICK_TEST:
            subs = sorted(df['subject'].unique())[:3]
            df   = df[df['subject'].isin(subs)]
            print(f"  subjects (quick): {subs}")

        print(f"  shape={df.shape} | subjects={df['subject'].nunique()} "
              f"| labels={df['label'].value_counts().to_dict()}")

        save_dir = MODEL_DIR / MODEL_TAG / exp_name
        save_dir.mkdir(parents=True, exist_ok=True)

        mdf, pdf = run_loso(df, exp_name=exp_name,
                            use_attention=USE_ATTENTION,
                            fold_save_dir=save_dir)
        mdf.insert(0, 'experiment', exp_name)
        mdf.insert(0, 'model', MODEL_TAG)
        all_metrics.append(mdf)
        all_predictions.append(pdf)

    except Exception as e:
        print(f"   {exp_name} failed: {e}")
        traceback.print_exc()

print("\n All experiments done")


Full grid (standard): 12 experiments

  CNN_LSTM · chest_acc
  shape=(975, 28) | subjects=15 | labels={0: 613, 1: 362}

Fold 1/15  test=S10
  shape=(1, 23)  train=908  test=67
  stopped epoch 39/100
  Acc=0.7313  F1=0.6667  AUC=0.8565  TP=18  FP=10  FN=8  TN=31

Fold 2/15  test=S11
  shape=(1, 23)  train=908  test=67
  stopped epoch 38/100
  Acc=0.8657  F1=0.8235  AUC=0.9714  TP=21  FP=5  FN=4  TN=37

Fold 3/15  test=S13
  shape=(1, 23)  train=909  test=66


  stopped epoch 46/100


  Acc=0.7424  F1=0.5143  AUC=0.8224  TP=9  FP=2  FN=15  TN=40

Fold 4/15  test=S14
  shape=(1, 23)  train=911  test=64
  stopped epoch 28/100
  Acc=0.5469  F1=0.6234  AUC=0.7344  TP=24  FP=29  FN=0  TN=11

Fold 5/15  test=S15
  shape=(1, 23)  train=909  test=66
  stopped epoch 32/100
  Acc=0.5909  F1=0.3721  AUC=0.7005  TP=8  FP=10  FN=17  TN=31

Fold 6/15  test=S16
  shape=(1, 23)  train=910  test=65
  stopped epoch 21/100
  Acc=0.9846  F1=0.9796  AUC=0.9990  TP=24  FP=1  FN=0  TN=40

Fold 7/15  test=S17
  shape=(1, 23)  train=908  test=67
  stopped epoch 24/100
  Acc=0.9254  F1=0.8936  AUC=1.0000  TP=21  FP=0  FN=5  TN=41

Fold 8/15  test=S2
  shape=(1, 23)  train=912  test=63
  stopped epoch 32/100
  Acc=0.8889  F1=0.8444  AUC=0.9630  TP=19  FP=3  FN=4  TN=37

Fold 9/15  test=S3
  shape=(1, 23)  train=911  test=64
  stopped epoch 33/100
  Acc=0.9844  F1=0.9796  AUC=0.9917  TP=24  FP=1  FN=0  TN=39

Fold 10/15  test=S4
  shape=(1, 23)  train=912  test=63
  stopped epoch 33/100
  Acc=

In [ ]:
# ── Cell · Save Results to Drive ────────────────────────────────────────────
if all_metrics:
    detailed_df = pd.concat(all_metrics, ignore_index=True)
    out_path    = OUTPUT_DIR / f'dl_{MODEL_TAG}_results_detailed.csv'
    detailed_df.to_csv(out_path, index=False)
    print(f" Metrics  → {out_path}  ({len(detailed_df)} rows)")

    print("\n── Mean metrics per experiment ──")
    print(detailed_df.groupby(['model', 'experiment'])
          [['f1','roc_auc','accuracy']].mean().round(3).to_string())

if all_predictions:
    curves_df  = pd.concat(all_predictions, ignore_index=True)
    curve_path = OUTPUT_DIR / f'dl_{MODEL_TAG}_predictions.csv'
    curves_df.to_csv(curve_path, index=False)
    print(f" Predictions → {curve_path}  ({len(curves_df)} rows)")
    print(f"   Columns: {list(curves_df.columns)}")

if not all_metrics and not all_predictions:
    print("  No results — check for errors in the Run cell above.")


---
## 💾 (Optional) Train & Save Production Model
Set `BEST_EXP` and `USE_ATTENTION` to match your best run.

In [ ]:
# ── Cell · Train & Save Production Model (all subjects) ─────────────────────
# After reviewing results above, set BEST_EXP to the top-performing experiment.

BEST_EXP    = 'chest_all'                                  # ✏️  edit me
BEST_CONFIG = {'chest': ['acc', 'eda', 'emg', 'heart', 'statistical']}  # ✏️

import joblib

print(f"Production training: {MODEL_TAG} / {BEST_EXP} on ALL subjects")

df_prod       = load_dataset(BEST_CONFIG)
feature_names = get_feature_names(df_prod)
X_raw         = df_prod[feature_names].values.astype(np.float32)
y             = df_prod['label'].values.astype(np.int32)

medians = np.nanmedian(X_raw, axis=0)
mask    = np.isnan(X_raw)
X_raw[mask] = np.take(medians, np.where(mask)[1])
scaler  = StandardScaler()
X_sc    = scaler.fit_transform(X_raw)
X_sc    = np.nan_to_num(X_sc, nan=0.0, posinf=0.0, neginf=0.0)
X_input = prepare_input(X_sc, feature_names)   # reshape for model

print(f"Full dataset: {df_prod.shape} | {df_prod['subject'].nunique()} subjects")

classes = np.unique(y)
cw      = compute_class_weight('balanced', classes=classes, y=y)

tf.keras.backend.clear_session()
prod_model = build_model(X_input.shape[1:])
prod_model.fit(
    X_input, y,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=dict(zip(classes.tolist(), cw.tolist())),
    validation_split=0.05,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1)],
    verbose=1,
)

save_dir = MODEL_DIR / MODEL_TAG / BEST_EXP
save_dir.mkdir(parents=True, exist_ok=True)
prod_model.save(str(save_dir / 'final_model.keras'))
joblib.dump(scaler, save_dir / 'scaler.joblib')
np.save(str(save_dir / 'medians.npy'), medians)
with open(save_dir / 'meta.json', 'w') as f:
    json.dump(dict(model=MODEL_TAG, experiment=BEST_EXP,
                   feature_names=feature_names, input_shape=list(X_input.shape[1:]),
                   n_subjects=int(df_prod['subject'].nunique()),
                   n_windows=int(len(df_prod)),
                   label_map={'0':'baseline','1':'stress'}), f, indent=2)

print(f"\n✅ Production model saved → {save_dir}")
print("   Files:", [p.name for p in save_dir.iterdir()])


In [ ]:
import joblib
import json
import gc
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler

# 1. Define all configurations upfront
EXPERIMENTS = {
    'chest_acc':         {'chest': ['acc']},
    'chest_eda':         {'chest': ['eda']},
    'chest_emg':         {'chest': ['emg']},
    'chest_heart':       {'chest': ['heart']},
    'chest_statistical': {'chest': ['statistical']},
    'wrist_acc':         {'wrist': ['acc']},
    'wrist_eda':         {'wrist': ['eda']},
    'wrist_heart':       {'wrist': ['heart']},
    'wrist_statistical': {'wrist': ['statistical']},
    'chest_all':         {'chest': ['acc', 'eda', 'emg', 'heart', 'statistical']},
    'wrist_all':         {'wrist': ['acc', 'eda', 'heart', 'statistical']},
    'multimodal':        {'chest': ['acc', 'eda', 'emg', 'heart', 'statistical'],
                          'wrist': ['acc', 'eda', 'heart', 'statistical']},
}

# 2. Sequential training loop with memory purging
for exp_name, exp_config in EXPERIMENTS.items():
    print("\n" + "="*80)
    print(f"🚀 Training: {MODEL_TAG} / {exp_name}")
    print("="*80)

    # --- Load & Preprocess Data ---
    df_prod = load_dataset(exp_config)
    feature_names = get_feature_names(df_prod)

    X_raw = df_prod[feature_names].values.astype(np.float32)
    y     = df_prod['label'].values.astype(np.int32)

    print(f"Data shape: {df_prod.shape} | Subjects: {df_prod['subject'].nunique()}")

    medians = np.nanmedian(X_raw, axis=0)
    mask    = np.isnan(X_raw)
    X_raw[mask] = np.take(medians, np.where(mask)[1])

    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_raw)
    X_sc   = np.nan_to_num(X_sc, nan=0.0, posinf=0.0, neginf=0.0)
    X_input = prepare_input(X_sc, feature_names)

    # --- Compute Weights & Build Model ---
    classes = np.unique(y)
    cw      = compute_class_weight('balanced', classes=classes, y=y)

    # Clear any residual graph data from previous loops before building
    tf.keras.backend.clear_session()

    prod_model = build_model(X_input.shape[1:])

    # --- Train Model ---
    prod_model.fit(
        X_input, y,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        class_weight=dict(zip(classes.tolist(), cw.tolist())),
        validation_split=0.05,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1
        )],
        verbose=1,
    )

    # --- Export Artifacts ---
    save_dir = MODEL_DIR / MODEL_TAG / exp_name
    save_dir.mkdir(parents=True, exist_ok=True)

    prod_model.save(str(save_dir / 'final_model.keras'))
    joblib.dump(scaler, save_dir / 'scaler.joblib')
    np.save(str(save_dir / 'medians.npy'), medians)

    with open(save_dir / 'meta.json', 'w') as f:
        json.dump(dict(
            model=MODEL_TAG, experiment=exp_name,
            feature_names=feature_names, input_shape=list(X_input.shape[1:]),
            n_subjects=int(df_prod['subject'].nunique()), n_windows=int(len(df_prod)),
            label_map={'0': 'baseline', '1': 'stress'}
        ), f, indent=2)

    print(f" Saved artifacts to {save_dir}")

    # ── CRITICAL COLAB MEMORY PURGE ─────────────────────────────────────────
    # Break down all large references and force immediate garbage collection
    del df_prod, X_raw, X_sc, X_input, y, medians, mask, prod_model, scaler
    tf.keras.backend.clear_session()
    gc.collect()
    # ────────────────────────────────────────────────────────────────────────

print("\n Done")


🚀 Training: cnn_lstm / chest_acc
Data shape: (975, 28) | Subjects: 15
Epoch 1/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 14s 43ms/step - accuracy: 0.7084 - auc: 0.8012 - loss: 0.5969 - val_accuracy: 0.5306 - val_auc: 0.9281 - val_loss: 0.6220
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8229 - auc: 0.9145 - loss: 0.4147 - val_accuracy: 0.6531 - val_auc: 0.9799 - val_loss: 0.5337
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8726 - auc: 0.9328 - loss: 0.3290 - val_accuracy: 0.6531 - val_auc: 0.9891 - val_loss: 0.5009
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8844 - auc: 0.9467 - loss: 0.2893 - val_accuracy: 0.7143 - val_auc: 0.9875 - val_loss: 0.5104
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9060 - auc: 0.9607 - loss: 0.2514 - val_accuracy: 0.7347 - val_auc: 1.0000 - val_loss: 0.4908
Epoch 6/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9104 - auc: 0.9669 - loss: 0.2309 - val_accuracy: 0.7347 - val